# Labor Rights RAG Assistant

### Project Goal:

This project implements a Retrieval-Augmented Generation (RAG) pipeline for answering legal questions based on the Guide to Employment Rights in Lebanon.

## Imports

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

import faiss
from sentence_transformers import SentenceTransformer

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Setup 

In [3]:
load_dotenv()

# Set API key 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")

# Define file path
DATA_DIR = "data"

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"{DATA_DIR} directory not found")

print("Environment loaded successfully.")

Environment loaded successfully.


### Pipeline Overview:

1. Document Loading & Chunking:

    - The PDF is loaded and split into overlapping text chunks.

    - Chunk size and overlap are chosen to balance context coverage and retrieval precision.

2. Embedding Generation

    - Each chunk is converted into a dense vector using a sentence-transformer model.

    - Embeddings are normalized to enable cosine similarity via inner product.

3. Vector Storage (FAISS)

    - Embeddings are stored in a FAISS IndexFlatIP index.

    - This enables efficient similarity search.

4. Query Processing

    - The user query is embedded.

    - Top-k most similar chunks are retrieved.

5. Answer Generation

    - Retrieved chunks are injected into a structured prompt.

    - The LLM (gpt-4.1-mini) generates a grounded answer.

    - If context is insufficient, the system explicitly refuses to hallucinate.

## PDF Loading

In [4]:
loader = PyPDFDirectoryLoader(DATA_DIR)
documents = loader.load()

if not documents:
    raise ValueError("No documents were loaded. Check your data folder.")

print(f"Total pages loaded (including blanks): {len(documents)}")

# Remove empty pages
documents = [doc for doc in documents if doc.page_content.strip()]

print(f"Pages after removing blanks: {len(documents)}")

if not documents:
    raise ValueError("All pages are empty after cleaning.")

# Show sample from first valid page
sample = documents[44]

print("\nSample Preview:")
print(sample.page_content[:500])

print("\nMetadata:", sample.metadata)

Total pages loaded (including blanks): 96
Pages after removing blanks: 95

Sample Preview:
CREDITS 
Acknowledgements:
Legal guide authored by Layal Abou Daher, legal consultant. Edited by Martin Clutterbuck/NRC. January 2024.
Design:
 
All photos @NRC and other external photographers as credited. 
Cover Photo: Palestinian Baker, (Photo: Racha El Daoi/NRC)
Back Cover Photo: Making Falafel, Bekaa (Photo: Racha El Dao/NRC)

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Windows)', 'creationdate': '2024-01-09T12:50:11+03:00', 'moddate': '2024-01-09T13:37:41+03:00', 'trapped': '/False', 'source': 'data\\Self-Employment-Guide.pdf', 'total_pages': 52, 'page': 1, 'page_label': '2'}


In [15]:
set(doc.metadata["source"] for doc in documents)

{'data\\Employment-Rights-Guide.pdf', 'data\\Self-Employment-Guide.pdf'}

## Chunking

In [5]:
import tiktoken

# Token counter
encoding = tiktoken.get_encoding("cl100k_base")

def token_length(text):
    return len(encoding.encode(text))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,       # tokens 
    chunk_overlap=80,
    length_function=token_length,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(documents)

# Add chunk ID
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    
print(f"Total chunks created: {len(chunks)}")
print(f"Created {len(chunks)} chunks from {len(documents)} documents")

print(f"\nSample Chunk:")
print(chunks[0].page_content[:300])
print(f"\nChunk metadata: {chunks[0].metadata}")

Total chunks created: 202
Created 202 chunks from 95 documents

Sample Chunk:
Acknowledgements. Legal guide authored by NRC and El Meouchi law firm ( https://www.elmeouchi.
com/  Chadia El Meouchi and Elie El Feghali), facilitated by TrustLaw, the Thomson Reuters Foundation’s 
global pro bono service. Edited by Martin Clutterbuck/NRC. Special thanks to Lianna Badamo, Sarah 
G

Chunk metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2023-03-16T15:35:55+03:00', 'moddate': '2023-03-18T09:26:14+03:00', 'trapped': '/False', 'source': 'data\\Employment-Rights-Guide.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2', 'chunk_id': 0}


## Generate Embeddings

In [6]:
# Initialize embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Extract text from chunks
texts = [chunk.page_content for chunk in chunks]

# Generate embeddings
embeddings = embedder.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True
)

dimension = embeddings.shape[1]

print(f"Embedding dimension: {dimension}")
print(f"Total embeddings: {embeddings.shape[0]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 251.76it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 7/7 [00:14<00:00,  2.09s/it]

Embedding dimension: 384
Total embeddings: 202


## Create FAISS Index

In [7]:
index = faiss.IndexFlatIP(dimension)  # use cosine similarity
index.add(embeddings.astype("float32"))

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 202 vectors.


## Save FAISS Index

In [8]:
import pickle

faiss.write_index(index, "index.faiss")

with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Vector store saved successfully.")

Vector store saved successfully.


## Retrieval and Answer Generation Functions

In [43]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def retrieve(query, k=3):
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = []
    for idx, score in zip(indices[0], scores[0]):
        metadata = chunks[idx].metadata
        results.append({
            "score": float(score),
            "source": metadata.get("source"),
            "page": metadata.get("page_label"),
            "content": chunks[idx].page_content
        })

    return results


def generate_answer(query, k=3):
    try:
        retrieved_docs = retrieve(query, k=k)

        if not retrieved_docs:
            return {
                "answer": "No relevant documents were retrieved.",
                "sources": []
            }

        context = "\n\n---\n\n".join(
            doc["content"] for doc in retrieved_docs
        )

        if not context.strip():
            return {
                "answer": "Retrieved documents contain no usable content.",
                "sources": []
            }

        prompt = f"""
You are a legal assistant answering questions strictly based on the provided context.

Rules:
- Only use the information in the context.
- If the answer is not in the context, say: "The guide does not provide this information."
- Be clear and structured.
- Do not invent laws or numbers.

Context:
{context}

Question:
{query}

Answer:
"""

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "You are a precise legal assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=500
        )

        if not response.choices:
            return {
                "answer": "Model returned no response.",
                "sources": []
            }

        answer = response.choices[0].message.content.strip()

        if "The guide does not provide this information." in answer:
             return {
        "answer": answer,
        "sources": []
            }

        sources = list(set(
            f"{doc['source']} (Page {doc['page']})"
            for doc in retrieved_docs
        ))

        return {
            "answer": answer,
            "sources": sources
        }

    except Exception as e:
        return {
            "answer": f"An error occurred during answer generation: {str(e)}",
            "sources": []
        }

## Test RAG Pipeline on Sample Questions

In [57]:
evaluation_questions = [

    # EMPLOYMENT RIGHTS GUIDE (17)
    "What is the legally mandated number of annual leave days?",
    "How does annual leave increase based on years of service?",
    "What is the official minimum wage?",
    "What are the maximum legal working hours per week?",
    "How is overtime compensated?",
    "What are the rules governing sick leave?",
    "Is maternity leave paid and for how long?",
    "What rights exist during the probation period?",
    "What prior notice is required before termination?",
    "What compensation is due in case of abusive dismissal?",
    "Under what conditions can employment be terminated due to force majeure?",
    "What are the employer's obligations regarding social security contributions?",
    "Are foreign workers entitled to social security benefits?",
    "Are employees protected from dismissal while on leave?",
    "What are the procedures for challenging abusive dismissal?",
    "Does the guide regulate cryptocurrency salary payments?",
    "Does the guide contain rules about remote work or work-from-home policies?",
    
    # SELF-EMPLOYMENT & SMALL BUSINESS GUIDE (12)
    "How does the guide define workers not linked to an employer?",
    "Are self-employed individuals covered by social security?",
    "Are foreigners eligible for social security benefits?",
    "When is a daily worker considered an employee versus self-employed?",
    "Must individuals conducting commercial activities register as traders?",
    "What activities are considered commercial in nature?",
    "Are workers who work for multiple employers covered by social security?",
    "Are self-employed individuals required to pay tax?",
    "Are self-employed individuals required to pay municipal fees, and on what basis are they calculated?",
    "Are small-scale daily labourers considered traders under commercial law?",
    "Does the guide set minimum pricing standards for freelance services?",
    "Does the guide provide government grants or startup funding schemes for self-employed workers?"
]

In [58]:
import pandas as pd

results = []

for question in evaluation_questions:
    output = generate_answer(question, k=3)

    results.append({
        "question": question,
        "answer": output["answer"],
        "sources": ", ".join(output["sources"])
    })

In [59]:
# For clean display

from IPython.display import display, Markdown

for r in results:
    display(Markdown(f"### Question\n{r['question']}"))
    display(Markdown(f"**Answer:**\n\n{r['answer']}"))
    display(Markdown(f"**Sources:** {r['sources']}"))
    display(Markdown("---"))

### Question
What is the legally mandated number of annual leave days?

**Answer:**

The legally mandated number of annual leave days according to Article 39 of the Labour Code is 15 working days, fully paid, provided that the employee has been employed by the employer for at least one year. 

However, while the Labour Code itself does not require an increase in annual leave based on years of service, the Ministry of Labour suggests the following minimum number of annual leave days based on length of service:

- 15 working days for employees employed between 1 to 5 years.
- 17 working days for employees employed between 5 to 10 years.
- 19 working days for employees employed between 10 to 15 years.
- 21 working days for employees employed for more than 15 years.

These suggested increases are based on Lebanon's membership in several Arab and International conventions.

**Sources:** data\Employment-Rights-Guide.pdf (Page 24), data\Employment-Rights-Guide.pdf (Page 16)

---

### Question
How does annual leave increase based on years of service?

...

---

### Question
Does the guide provide government grants or startup funding schemes for self-employed workers?

**Answer:**

The guide does not provide this information.

**Sources:** 

---

## Evaluation

In [61]:
ground_truth = {

    # EMPLOYMENT RIGHTS GUIDE 

    "What is the legally mandated number of annual leave days?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [16, 24]
    },

    "How does annual leave increase based on years of service?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [16]
    },

    "What is the official minimum wage?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [13]
    },

    "What are the maximum legal working hours per week?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [15]
    },

    "How is overtime compensated?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [15]
    },

    "What are the rules governing sick leave?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [17]
    },

    "Is maternity leave paid and for how long?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [17]
    },

    "What rights exist during the probation period?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [12]
    },

    "What prior notice is required before termination?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [21]
    },

    "What compensation is due in case of abusive dismissal?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [21]
    },

    "Under what conditions can employment be terminated due to force majeure?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [22, 23]
    },

    "What are the employer's obligations regarding social security contributions?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [14, 15, 34]
    },

    "Are foreign workers entitled to social security benefits?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [15, 34]
    },

    "Are employees protected from dismissal while on leave?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [16, 17, 24]
    },

    "What are the procedures for challenging abusive dismissal?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [24]
    },

    "Does the guide regulate cryptocurrency salary payments?": {
    "document": None,
    "pages": []
    },

    "Does the guide contain rules about remote work or work-from-home policies?": {
    "document": None,
    "pages": []
    },


    # SELF-EMPLOYMENT GUIDE
    "How does the guide define workers not linked to an employer?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [24]
    },

    "Are self-employed individuals covered by social security?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [26, 27]
    },

    "Are foreigners eligible for social security benefits?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [27]
    },

    "When is a daily worker considered an employee versus self-employed?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [27]
    },

    "Must individuals conducting commercial activities register as traders?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [29, 30]
    },

    "What activities are considered commercial in nature?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [28, 29]
    },

    "Are workers who work for multiple employers covered by social security?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [26, 27]
    },

    "Are self-employed individuals required to pay tax?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [32, 33]
    },

    "Are self-employed individuals required to pay municipal fees, and on what basis are they calculated?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [34]
    },

    "Are small-scale daily labourers considered traders under commercial law?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [29, 30]
    },

    "Does the guide set minimum pricing standards for freelance services?": {
    "document": None,
    "pages": []
    },

    "Does the guide provide government grants or startup funding schemes for self-employed workers?": {
    "document": None,
    "pages": []
    },
}

In [62]:
# check for missing questions
for q in evaluation_questions:
    if q not in ground_truth:
        print("Missing in ground_truth:", q)

#### Precision@K

In [63]:
def compute_precision_at_k(evaluation_questions, ground_truth, k=3):
    total_retrieved = 0
    total_relevant = 0

    for question in evaluation_questions:

        gt = ground_truth[question]
        gt_doc = gt["document"]
        gt_pages = set(str(p) for p in gt["pages"])

        retrieved_docs = retrieve(question, k=k)

        # Remove duplicate (doc, page) pairs
        retrieved_pairs = set(
            (
                doc["source"].split("\\")[-1],
                str(doc["page"])
            )
            for doc in retrieved_docs
        )

        total_retrieved += len(retrieved_pairs)

        for doc_name, page in retrieved_pairs:
            if gt_doc is not None and doc_name == gt_doc and page in gt_pages:
                total_relevant += 1

    return total_relevant / total_retrieved if total_retrieved > 0 else 0


precision_at_3 = compute_precision_at_k(
    evaluation_questions,
    ground_truth,
    k=3
)

print(f"Precision@3: {precision_at_3:.2f}")


Precision@3: 0.39


#### Recall@K

In [65]:
def compute_recall_at_k(evaluation_questions, ground_truth, k=3):
    total_questions = 0
    questions_with_hit = 0

    for question in evaluation_questions:

        gt = ground_truth[question]
        gt_doc = gt["document"]
        gt_pages = set(str(p) for p in gt["pages"])

        if gt_doc is None:
            continue  # skip negative questions for recall

        total_questions += 1

        retrieved_docs = retrieve(question, k=k)

        retrieved_pairs = set(
            (
                doc["source"].split("\\")[-1],
                str(doc["page"])
            )
            for doc in retrieved_docs
        )

        for doc_name, page in retrieved_pairs:
            if doc_name == gt_doc and page in gt_pages:
                questions_with_hit += 1
                break

    return questions_with_hit / total_questions if total_questions > 0 else 0

recall_at_3 = compute_recall_at_k(
    evaluation_questions,
    ground_truth,
    k=3
)

print(f"Recall@3: {recall_at_3:.2f}")

Recall@3: 0.96


#### Hit@K

In [66]:
def compute_hit_at_k(evaluation_questions, ground_truth, k=3):
    hits = 0
    total = 0

    for question in evaluation_questions:

        gt = ground_truth[question]
        gt_doc = gt["document"]
        gt_pages = set(str(p) for p in gt["pages"])

        if gt_doc is None:
            continue

        total += 1

        retrieved_docs = retrieve(question, k=k)

        for doc in retrieved_docs:
            doc_name = doc["source"].split("\\")[-1]
            page = str(doc["page"])

            if doc_name == gt_doc and page in gt_pages:
                hits += 1
                break

    return hits / total if total > 0 else 0

hit_at_3 = compute_hit_at_k(
    evaluation_questions,
    ground_truth,
    k=3
)

print(f"Hit@3: {hit_at_3:.2f}")

Hit@3: 0.96


In [67]:
for k in [1, 3, 5]:
    print(f"\nK = {k}")
    print(f"Hit@{k}: {compute_hit_at_k(evaluation_questions, ground_truth, k=k):.2f}")
    print(f"Recall@{k}: {compute_recall_at_k(evaluation_questions, ground_truth, k=k):.2f}")


K = 1
Hit@1: 0.76
Recall@1: 0.76

K = 3
Hit@3: 0.96
Recall@3: 0.96

K = 5
Hit@5: 0.96
Recall@5: 0.96


### Retrieval Evaluation Summary:

**Metrics Used:**

- **Precision@K** - Proportion of retrieved pages in the top K that are relevant. Measures retrieval cleanliness (noise level).

- **Recall@K** - Measures whether the retriever successfully retrieves the relevant ground-truth page(s) within the top K results.

- **Hit@K** - Measures whether at least one correct source page is retrieved in the top K results.

**Interpretation:**

- At K=1, the retriever returns the correct page as the top result in 76% of cases.

- At K=3, relevant pages are retrieved in 96% of questions.

- Increasing K to 5 does not improve performance, indicating that relevant pages are already retrieved within the top 3.

**Conclusion:**

The retriever performs strongly.

While Precision@3 was lower due to extra non-relevant pages being retrieved, Hit@K and Recall@K demonstrate that the system reliably retrieves the correct source context for nearly all questions.

### LLM Answer Evaluation:

- Total Evaluation Questions: 29

- Retrieval Top-k: 3

- LLM Model: gpt-4.1-mini

**Observations:**

- The system produces structured and grounded legal answers.

- Irrelevant questions are correctly rejected with:

    "The guide does not provide this information."

- No hallucinated legal numbers were observed during evaluation.

This indicates effective retrieval grounding and prompt control.

### Limitations

- Evaluation conducted on a relatively small, manually selected dataset.

- Retrieval relies solely on dense vector search (no reranking).

- Fixed top-k retrieval strategy without adaptive selection..

- Evaluation focuses on retrieval metrics; answer quality assessed manually.

### Future Improvements

- Introduce reranking for improved precision.

- Implement hybrid search (BM25 + dense retrieval).

- Experiment with adaptive top-K selection.

- Add automated answer-level evaluation (e.g., LLM-as-judge or semantic similarity scoring).